# Conjugate Gradient — Interactive Experiments

Compare Conjugate Gradient with SciPy for real symmetric
positive-definite and complex Hermitian positive-definite systems.

**Instructions:** Choose the parameters and click the corresponding
experiment button. Results and plots appear below each experiment.


In [ ]:
from pathlib import Path
import sys
import time

# Find the repository root when launched locally or with Binder.
root = Path.cwd().resolve()
for candidate in (root, *root.parents):
    if (candidate / "krylov_methods").is_dir():
        root = candidate
        break
else:
    raise RuntimeError("Start Jupyter from within the repository checkout.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from scipy.sparse.linalg import cg as scipy_cg

from krylov_methods.conjugate_gradient_solver import solve_conjugate_gradient
from krylov_methods.notebooks.notebook_utils import (
    spd_problem,
    hermitian_pd_problem,
    block_jacobi_inverse,
    residual_plot,
    error_plot,
    report,
)


In [ ]:
kind = widgets.Dropdown(
    options=[("Real SPD", "real"), ("Complex Hermitian PD", "complex")],
    description="System",
)
size = widgets.IntSlider(value=30, min=5, max=100, step=5, description="Size n")
condition = widgets.FloatLogSlider(
    value=1e4, base=10, min=1, max=7, step=0.5, description="Cond(A)"
)
tol = widgets.FloatLogSlider(
    value=1e-9, base=10, min=-12, max=-5, step=1, description="rtol"
)
budget = widgets.IntSlider(
    value=500, min=20, max=1500, step=20, description="Max iterations"
)
preconds = widgets.Dropdown(
    options=["none", "jacobi", "ssor", "cholesky", "ic0", "custom (block Jacobi)"],
    value="none",
    description="Precond",
    layout=widgets.Layout(width="340px"),
)
button = widgets.Button(description="Run experiment", button_style="success")
output = widgets.Output()
cg_residual_output = widgets.Output()
cg_error_output = widgets.Output()


def run(_):
    with output:
        clear_output(wait=True)
        cg_residual_output.clear_output(wait=True)
        cg_error_output.clear_output(wait=True)
        from contextlib import redirect_stdout
        from io import StringIO
        from html import escape

        report_buffer = StringIO()
        with redirect_stdout(report_buffer):
            A, b, x_true = (
                spd_problem(size.value, condition.value)
                if kind.value == "real"
                else hermitian_pd_problem(size.value, condition.value)
            )
            descriptor = (
                "real symmetric positive definite"
                if kind.value == "real"
                else "complex Hermitian positive definite"
            )
            threshold = max(1e-12, tol.value * np.linalg.norm(b))
            eigs = np.linalg.eigvalsh(A)

            print("SYSTEM INFORMATION")
            print(f"Matrix A: dense {descriptor} ({size.value} × {size.value})")
            print(f"Hermitian error ||A-Aᴴ||₂: {np.linalg.norm(A - A.conj().T, ord=2):.2e}")
            print(f"Spectral interval: [{eigs[0]:.3e}, {eigs[-1]:.3e}]")
            print(f"2-norm condition number: {np.linalg.cond(A):.3e}")
            print(f"||b||₂ = {np.linalg.norm(b):.3e}; ||x_exact||₂ = {np.linalg.norm(x_true):.3e}")
            print(f"Initial x₀ = 0; ||r₀||₂ = {np.linalg.norm(b):.3e}")
            print(f"Stop when ||b-Ax||₂ ≤ max(1e-12, rtol ||b||₂) = {threshold:.3e}")
            print(f"Iteration budget = {budget.value}\n")

            histories = {}
            error_histories = {}
            timings = {}
            for name in (preconds.value,):
                if name == "ic0" and kind.value == "complex":
                    print("Skipping IC(0): available only for real SPD matrices.\n")
                    continue

                custom = name == "custom (block Jacobi)"
                preconditioner_name = "custom" if custom else None if name == "none" else name

                # Custom setup is external; built-in preconditioners are built
                # inside solve_conjugate_gradient and counted in solver-call time.
                setup_start = time.perf_counter()
                inverse_action = block_jacobi_inverse(A) if custom else None
                external_setup_time = time.perf_counter() - setup_start

                solve_start = time.perf_counter()
                result = solve_conjugate_gradient(
                    A, b, rtol=tol.value, atol=1e-12,
                    max_iter=budget.value,
                    preconditioner=preconditioner_name,
                    M=inverse_action,
                    store_history=True,
                )
                solver_call_time = time.perf_counter() - solve_start
                timings[name] = (external_setup_time, solver_call_time)

                report(
                    "Our CG: " + name, result.solution, x_true, b, A,
                    result.iterations, result.converged,
                )
                error = result.solution - x_true
                energy_error = np.sqrt(max(0.0, np.vdot(error, A @ error).real))
                relative_residual = np.linalg.norm(b - A @ result.solution) / np.linalg.norm(b)
                print(f"  Relative residual: {relative_residual:.3e}")
                print(f"  Energy-norm error: {energy_error:.3e}")
                print(f"  External setup time: {external_setup_time:.6f} s")
                print(f"  Solver call time: {solver_call_time:.6f} s")
                print(f"  Status: {result.message}\n")

                histories["CG / " + name] = result.residual_norms
                error_histories["CG / " + name] = [
                    np.linalg.norm(xk - x_true) / np.linalg.norm(x_true)
                    for xk in result.history
                ]

            # Compare against SciPy CG, recording both metrics at each iteration.
            scipy_history = [np.linalg.norm(b)]
            scipy_errors = [1.0]  # x0 = 0, hence relative error = 1.

            def callback(xk):
                scipy_history.append(np.linalg.norm(b - A @ xk))
                scipy_errors.append(np.linalg.norm(xk - x_true) / np.linalg.norm(x_true))

            scipy_start = time.perf_counter()
            x, info = scipy_cg(
                A, b, rtol=tol.value, atol=1e-12,
                maxiter=budget.value, callback=callback,
            )
            scipy_time = time.perf_counter() - scipy_start
            report("SciPy CG", x, x_true, b, A, len(scipy_history) - 1, info == 0)
            print(f"  Solver call time: {scipy_time:.6f} s")
            print(f"  SciPy info={info} (0 means converged).\n")
            histories["SciPy CG"] = scipy_history
            error_histories["SciPy CG"] = scipy_errors

            with cg_residual_output:
                residual_plot(
                    histories, threshold,
                    title="CG — Residual convergence (" + descriptor + ")",
                    figsize=(4.8, 3.2),
            )
            with cg_error_output:
                error_plot(
                    error_histories,
                    title="CG — Relative solution-error convergence",
                    figsize=(4.8, 3.2),
            )

            if preconds.value == "cholesky":
                print("Note: Exact Cholesky factors A; it is a direct-solve reference, "
                      "not a cheap preconditioner for large systems.")
            if preconds.value == "custom (block Jacobi)":
                print("Custom inverse action: SPD diagonal blocks of size ≤ 4, "
                      "solved using triangular systems.")
            print("Timing note: built-in preconditioner setup is included in solver-call time; "
                  "only custom block-Jacobi has separately measured setup time.")



        report_text = report_buffer.getvalue()
        if report_text:
            report_html = widgets.HTML(
                value=(
                    "<pre style='white-space:pre-wrap;"
                    "font-size:12px;overflow-wrap:anywhere;'>"
                    + escape(report_text)
                    + "</pre>"
                )
            )
            details = widgets.Accordion(children=[report_html])
            details.set_title(0, 'System information & solver details')
            details.selected_index = None
            display(details)

button.on_click(run)
# Both experiments are displayed together in the final dashboard cell.


In [ ]:
kappa_slider = widgets.FloatLogSlider(value=1e10, base=10, min=2, max=12,
                                               step=1, description="Cond(A)")
absolute_tol = widgets.FloatLogSlider(value=1e-8, base=10, min=-12, max=-3,
                                      step=1, description="atol")
ill_button = widgets.Button(description="Explore residual vs error", button_style="info")
ill_output = widgets.Output()
diagnostic_plot_output = widgets.Output()

def explore(_):
    with ill_output:
        clear_output(wait=True)
        diagnostic_plot_output.clear_output(wait=True)
        from contextlib import redirect_stdout
        from io import StringIO
        from html import escape

        report_buffer = StringIO()
        with redirect_stdout(report_buffer):
            kappa = kappa_slider.value
            A = np.diag([1.0, 1.0 / kappa])
            exact = np.ones(2)
            b = A @ exact
            x0 = np.array([1.0, 0.0])
            r0 = b - A @ x0
            initial_error = np.linalg.norm(exact - x0)
            result = solve_conjugate_gradient(A, b, x0=x0, rtol=0.0,
                                              atol=absolute_tol.value, max_iter=2)
            final_error = np.linalg.norm(exact - result.solution)
            final_residual = np.linalg.norm(b - A @ result.solution)
            print(f"A = diag(1, 1/{kappa:.1e}); cond₂(A) = {np.linalg.cond(A):.1e}")
            print(f"Exact x*={exact}; initial x₀={x0}; b={b}")
            print(f"Initial ||r₀||₂={np.linalg.norm(r0):.3e}, ||e₀||₂={initial_error:.3e}")
            print(f"Selected absolute tolerance = {absolute_tol.value:.3e}")
            print(f"CG iterations = {result.iterations}; converged = {result.converged}")
            print(f"Final ||b-Ax||₂={final_residual:.3e}; final ||x*-x||₂={final_error:.3e}")
            print("Compare ||e||₂ to the bound ||A⁻¹||₂ ||r||₂ =",
                  f"{kappa * final_residual:.3e}")
            labels = ["Residual norm", "Solution error", "Absolute tolerance"]
            values = [max(final_residual, np.finfo(float).tiny),
                      max(final_error, np.finfo(float).tiny), absolute_tol.value]
            with diagnostic_plot_output:
                fig, ax = plt.subplots(figsize=(4.8, 3.2))
                ax.bar(labels, values, color=["#3182bd", "#e6550d", "#777777"])
                ax.set_yscale("log")
                ax.set_ylabel("Magnitude (log scale)")
                ax.set_title("Small residual ≠ small forward error")
                ax.tick_params(axis="x", rotation=10)
                fig.tight_layout()
                plt.show()


        report_text = report_buffer.getvalue()
        if report_text:
            report_html = widgets.HTML(
                value=(
                    "<pre style='white-space:pre-wrap;"
                    "font-size:12px;overflow-wrap:anywhere;'>"
                    + escape(report_text)
                    + "</pre>"
                )
            )
            details = widgets.Accordion(children=[report_html])
            details.set_title(0, 'Numerical experiment details')
            details.selected_index = None
            display(details)

ill_button.on_click(explore)



# -------------------------------------------------------
# Compact interactive dashboard
# -------------------------------------------------------

cg_status = widgets.HTML()
diagnostic_status = widgets.HTML()

cg_parameters = (kind, size, condition, tol, budget, preconds)
diagnostic_parameters = (kappa_slider, absolute_tol)

cg_last_run = None
diagnostic_last_run = None


def current_values(parameters):
    return tuple(widget.value for widget in parameters)


def status_html(is_current):
    if is_current:
        return (
            "<span style='color:#22863a;font-weight:600;'>"
            "● Simulation up to date</span>"
        )
    return (
        "<span style='color:#b36b00;font-weight:600;'>"
        "● Parameters changed — rerun required</span>"
    )


def update_cg_status(change=None):
    cg_status.value = (
        ""
        if cg_last_run is None
        else status_html(current_values(cg_parameters) == cg_last_run)
    )


def update_diagnostic_status(change=None):
    diagnostic_status.value = (
        ""
        if diagnostic_last_run is None
        else status_html(
            current_values(diagnostic_parameters) == diagnostic_last_run
        )
    )


def execute_cg(change=None):
    global cg_last_run
    cg_last_run = None
    update_cg_status()
    run(None)

    # Reveal the two CG plots and their numerical report.
    plots_row.children[0].layout.display = ""
    plots_row.children[1].layout.display = ""
    cg_details_panel.layout.display = ""

    cg_last_run = current_values(cg_parameters)
    update_cg_status()


def execute_diagnostic(change=None):
    global diagnostic_last_run
    diagnostic_last_run = None
    update_diagnostic_status()
    explore(None)

    # Reveal only the residual/error plot and its numerical report.
    plots_row.children[2].layout.display = ""
    diagnostic_details_panel.layout.display = ""

    diagnostic_last_run = current_values(diagnostic_parameters)
    update_diagnostic_status()


# Replace the original button callbacks with status-aware callbacks.
button.on_click(run, remove=True)
ill_button.on_click(explore, remove=True)

button.on_click(execute_cg)
ill_button.on_click(execute_diagnostic)

for parameter in cg_parameters:
    parameter.observe(update_cg_status, names="value")

for parameter in diagnostic_parameters:
    parameter.observe(update_diagnostic_status, names="value")

update_cg_status()
update_diagnostic_status()


# -------------------------------------------------------
# Controls
# -------------------------------------------------------

cg_controls = widgets.VBox(
    [
        widgets.HTML("<h3>Conjugate Gradient Convergence</h3>"),
        kind,
        size,
        condition,
        tol,
        budget,
        preconds,
        button,
        cg_status,
    ],
    layout=widgets.Layout(
        flex="2 1 0%",
        min_width="320px",
        padding="10px",
        border="1px solid #dddddd",
    ),
)

diagnostic_controls = widgets.VBox(
    [
        widgets.HTML("<h3>Residual vs. Solution Error</h3>"),
        kappa_slider,
        absolute_tol,
        ill_button,
        diagnostic_status,
    ],
    layout=widgets.Layout(
        flex="1 1 0%",
        min_width="280px",
        padding="10px",
        border="1px solid #dddddd",
    ),
)

controls_row = widgets.HBox(
    [cg_controls, diagnostic_controls],
    layout=widgets.Layout(
        width="100%",
        flex_flow="row wrap",
        gap="10px",
        align_items="stretch",
    ),
)


# -------------------------------------------------------
# Three plots
# -------------------------------------------------------

def plot_panel(title, plot_output):
    return widgets.VBox(
        [
            widgets.HTML(
                "<div style='font-weight:600;text-align:center;'>"
                + title + "</div>"
            ),
            plot_output,
        ],
        layout=widgets.Layout(
            flex="1 1 0%",
            min_width="280px",
            max_width="100%",
            padding="5px",
            overflow="hidden",
        ),
    )


plots_row = widgets.HBox(
    [
        plot_panel("Residual convergence", cg_residual_output),
        plot_panel("Relative solution error", cg_error_output),
        plot_panel("Residual vs. error", diagnostic_plot_output),
    ],
    layout=widgets.Layout(
        width="100%",
        flex_flow="row wrap",
        gap="5px",
        align_items="flex-start",
    ),
)


# -------------------------------------------------------
# Expandable numerical reports
# -------------------------------------------------------

cg_details_panel = widgets.VBox(
    [output],
    layout=widgets.Layout(
        flex="2 1 0%",
        min_width="320px",
        padding="5px",
    ),
)

diagnostic_details_panel = widgets.VBox(
    [ill_output],
    layout=widgets.Layout(
        flex="1 1 0%",
        min_width="280px",
        padding="5px",
    ),
)

details_row = widgets.HBox(
    [cg_details_panel, diagnostic_details_panel],
    layout=widgets.Layout(
        width="100%",
        flex_flow="row wrap",
        gap="10px",
        align_items="flex-start",
    ),
)


# -------------------------------------------------------
# Final dashboard
# -------------------------------------------------------

dashboard = widgets.VBox(
    [
        widgets.HTML("""
        <style>
        .cg-compact-dashboard .output_area img,
        .cg-compact-dashboard .jp-OutputArea-output img {
            max-width: 100% !important;
            height: auto !important;
        }
        </style>
        """),
        controls_row,
        plots_row,
        details_row,
    ],
    layout=widgets.Layout(
        width="100%",
        gap="10px",
    ),
)

# Initially hide every plot and numerical-report panel.
for panel in plots_row.children:
    panel.layout.display = "none"

cg_details_panel.layout.display = "none"
diagnostic_details_panel.layout.display = "none"

dashboard.add_class("cg-compact-dashboard")

display(dashboard)


# -------------------------------------------------------
# Experiments run only when visitors click their buttons.
# No plots or numerical reports appear on startup.
